# keras vs pytorch

same mlp on fashion-mnist in both. comparing the api ergonomics.


In [3]:
# keras side
import tensorflow as tf
(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.fashion_mnist.load_data()
x_tr = x_tr.reshape(-1, 784) / 255.0
x_te = x_te.reshape(-1, 784) / 255.0


In [4]:
model_k = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10),
])
model_k.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])


In [5]:
# keras training loop
model_k.fit(x_tr, y_tr, epochs=5, batch_size=128)


In [6]:
# pytorch side
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

x_tr_t = torch.tensor(x_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
loader = DataLoader(TensorDataset(x_tr_t, y_tr_t), batch_size=128, shuffle=True)


In [7]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

model_p = MLP()
opt = torch.optim.Adam(model_p.parameters(), lr=1e-3)


In [8]:
# train pytorch
for ep in range(5):
    for x,y in loader:
        opt.zero_grad()
        F.cross_entropy(model_p(x), y).backward()
        opt.step()


In [9]:
# observation: keras is more terse for vanilla cases. pytorch is clearer when the loop matters.
# both end up at very similar val accuracy (around 88%).


hp tweak.